# Lab 4 : Chunking, the choice that decides whether RAG works

*W3 RAG Part 1 · Utrains LLMOps 8-Week Course*

Run each cell in order. Read the output. Move to the next.

See the matching slide in this week's concepts deck for the real-world story this lab teaches.


## What we are achieving in this lab

**Objective.** A long document is not one embedding. **Split it, then embed each piece.** See why, on the real `handbook.txt`.

**Prerequisites.** Labs 1–3 finished. Same OpenAI key. You already have `embed` and `cosine`.

**The line so far.**

| Lab | What you can do |
|-----|-----------------|
| 1 | Turn a sentence into a vector |
| 2 | Score two vectors with cosine |
| 3 | Search a pile of **short** snippets |
| 4 (this lab) | Those snippets have to come from somewhere. Split a real file, then embed. |

**What you will do.**

1. Load `handbook.txt` and print it.
2. Embed the **whole file** vs each **section**. Same question. Read the cosine scores.
3. Split the file with `RecursiveCharacterTextSplitter` and **print every chunk**.
4. Rank those chunks with Lab 2 cosine, at two sizes.

**Cost.** A few embedding calls. Fractions of a cent.


## Where this sits after Labs 1–3

Lab 1: one string → one vector. A word and a paragraph both become **1536** numbers.

Lab 3 searched six short snippets. A real handbook is one long string. If you embed the whole file, those 1536 numbers mix **every** topic: reimbursements, PTO, parental leave. Cosine against a train-ticket question then looks at a blur.

**Chunking** is the fix: cut the file into pieces, embed each piece (Lab 1), score with cosine (Lab 2). Same search as Lab 3. The new skill is producing the snippets.

Order that matters: **split, then embed.** You cannot split a vector back into paragraphs.


### Step 1. Load the key and the handbook

If the key cell fails, fix `.env` from Lab 1 first. The handbook sits next to this notebook.


In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError(
        "Missing OPENAI_API_KEY. Copy .env.example to .env at the repository root, "
        "paste the key, and restart the kernel."
    )

print("OPENAI_API_KEY : set")


In [ ]:
from pathlib import Path

candidates = [Path("handbook.txt"), Path("week03") / "handbook.txt"]
path = None
for p in candidates:
    if p.exists():
        path = p
        break
if path is None:
    raise FileNotFoundError(
        "handbook.txt not found. Open this notebook from week03/, "
        "or start Jupyter from the repository root."
    )

HANDBOOK = path.read_text(encoding="utf-8")
print("file      :", path)
print("characters:", len(HANDBOOK))
print()
print(HANDBOOK)


### Step 2. Same embeddings and cosine as Labs 1–2


In [ ]:
import numpy as np
from langchain_openai import OpenAIEmbeddings

EMBED_MODEL = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model=EMBED_MODEL)


def embed(text: str) -> np.ndarray:
    # Lab 1: one string -> one vector.
    return np.asarray(embeddings.embed_query(text), dtype=float)


def cosine(a: np.ndarray, b: np.ndarray) -> float:
    # Lab 2: how close two arrows are. Higher = closer in meaning.
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


QUERY = "How do I get reimbursed for a $300 train ticket?"
print("Question:", QUERY)
print("Model   :", EMBED_MODEL)


### Step 3. Why not one vector for the whole file

Embed the full handbook. Embed each `##` section. Score all of them against the same question.

Watch the output: the whole file is **one** 1536-long vector (Lab 1). Reimbursements should beat the whole file. Parental leave should sit lower.


In [ ]:
q_vec = embed(QUERY)
whole_vec = embed(HANDBOOK)

print("Question:", QUERY)
print()
print("Whole handbook")
print("  vector length:", len(whole_vec), "  (Lab 1: always 1536 for this model)")
print("  cosine       :", round(cosine(q_vec, whole_vec), 3))
print()

# Split on headings so each section is one piece — still the real file, not toy strings.
parts = HANDBOOK.split("\n## ")
print("Each ## section vs the same question")
print()
for part in parts:
    heading = part.split("\n", 1)[0]
    heading = heading.replace("#", "").strip()
    score = cosine(q_vec, embed(part))
    print(round(score, 3), " ", heading)


The reimbursement section should be highest. The whole file sits somewhere in the middle: it *contains* the answer and also four unrelated sections, so the vector is mixed.

That is why we chunk, then embed.


### Step 4. Split the file and print the chunks

`RecursiveCharacterTextSplitter` tries to cut on paragraph breaks, then lines, then spaces — not in the middle of a word.

`chunk_size` is characters. `chunk_overlap` copies a little of the previous chunk onto the next one, so a sentence on the cut is not lost.

Step 1 printed the length. We try **200** (too small), **500** (usable), and **2000** (bigger than this file = still one piece).


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


def show_chunks(chunk_size: int, chunk_overlap: int) -> list[str]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    chunks = splitter.split_text(HANDBOOK)
    print("chunk_size =", chunk_size, "  overlap =", chunk_overlap, "  ->", len(chunks), "chunks")
    print()
    for i, text in enumerate(chunks):
        print("--- chunk", i, " (", len(text), "chars) ---")
        print(text)
        print()
    return chunks


print("===== too small =====")
small = show_chunks(200, 40)

print("===== usable =====")
medium = show_chunks(500, 50)

print("===== bigger than the file =====")
huge = show_chunks(2000, 0)


Read the printed chunks.

- **200:** Section 3 often splits. "Upload your receipt" can land in one chunk and "Above $500 requires approval" in the next. A $300 ticket needs that second sentence.
- **500:** A section usually fits. This is the size we want for this file.
- **2000:** One chunk. You are back to embedding the whole handbook.


### Step 5. Lab 3 cosine, on the chunks you just made

Same question. Embed the chunks (`embed_documents`, Lab 1). Score with cosine (Lab 2). Highest first.

No vector store. This is still a list and a loop.


In [ ]:
def rank_chunks(label: str, chunks: list[str]) -> None:
    print("===== ", label, " (", len(chunks), " chunks) =====", sep="")
    print("Question:", QUERY)
    print()

    # Lab 1: many strings at index time.
    vecs = embeddings.embed_documents(chunks)
    q_vec = embed(QUERY)

    scored = []
    for i, text in enumerate(chunks):
        score = cosine(q_vec, np.asarray(vecs[i], dtype=float))
        scored.append((score, i, text))
        preview = text.replace("\n", " ")
        if len(preview) > 90:
            preview = preview[:90] + "..."
        print(round(score, 3), "  chunk", i, " ", preview)

    print()
    scored.sort(reverse=True)
    best_score, best_i, best_text = scored[0]
    print("Highest: chunk", best_i, "  cosine", round(best_score, 3))
    print()
    print(best_text)
    print()


rank_chunks("chunk_size = 200", small)
rank_chunks("chunk_size = 500", medium)
rank_chunks("chunk_size = 2000 (whole file)", huge)


On **200**, the top chunk may be about reimbursements but miss the $500 rule if that sentence is in the next chunk.

On **500**, the top chunk should be the reimbursement section, including the $500 line.

On **2000**, the "top chunk" is the whole handbook. Cosine can still look fine. You did not isolate the answer.

## What you should be able to explain

> "I split a document into chunks, then I embed each chunk. I do not embed the whole file as one vector."

> "Too small: a rule and its neighbour land in different chunks. Too big: I am back to one mixed vector."

> "After chunking, search is the same cosine loop as Lab 3."

**Lab 5** puts those chunk vectors in a store so you do not run the `for` loop yourself.
